In [3]:
import sqlite3
import pandas as pd

In [4]:
con = sqlite3.connect('../data/AdventureWorks.db')
cur = con.cursor()

---
**Tables**

In [5]:
cur.execute('select name from sqlite_master where type = "table"').fetchall()

[('Calendar_Lookup',),
 ('Customer_Lookup',),
 ('Product_Categories_Lookup',),
 ('Product_Lookup',),
 ('Product_Subcategories_Lookup',),
 ('Returns_Data',),
 ('Territory_Lookup',),
 ('Sales_Data_2020',),
 ('Sales_Data_2021',),
 ('Sales_Data_2022',),
 ('Sales_Data',)]

In [11]:
%load_ext sql
%sql sqlite:///../data/AdventureWorks.db
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [16]:
%sql select * from Sales_Data limit 1;

 * sqlite:///../data/AdventureWorks.db
Done.


OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2020-01-01,2019-09-21,SO45080,332,14657,1,1,1


In [15]:
%sql select * from Product_Lookup limit 1;

 * sqlite:///../data/AdventureWorks.db
Done.


ProductKey,ProductSubcategoryKey,ProductSKU,ProductName,ModelName,ProductDescription,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice
214,31,HL-U509-R,"Sport-100 Helmet, Red",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Red,0,0,13.0863,34.99


---
**Orders Daily Revenue**

In [42]:
%%sql
select OrderDate,
round(OrderQuantity * ProductCost,2) as order_cost,
round(OrderQuantity * ProductPrice,2) as order_price,
round(sum((OrderQuantity * ProductPrice) - (OrderQuantity * ProductCost)),2) as order_revenue
from Sales_Data as s
inner join Product_Lookup as p
on s.ProductKey = p.ProductKey
group by OrderDate
limit 2;

 * sqlite:///../data/AdventureWorks.db
Done.


OrderDate,order_cost,order_price,order_revenue
2020-01-01,413.15,699.1,3455.78
2020-01-02,2171.29,3578.27,5627.9


---
**Running Total Monthly Revenue**

In [52]:
%%sql
with orders_revenue as
(select OrderDate,
round(sum((OrderQuantity * ProductPrice) - (OrderQuantity * ProductCost)),2) as order_revenue
from Sales_Data as s
inner join Product_Lookup as p
on s.ProductKey = p.ProductKey
group by OrderDate)

select *,
sum(order_revenue) over
(partition by strftime('%Y-%m', OrderDate)
order by OrderDate) as total_revenue
from orders_revenue
where strftime('%d', OrderDate) between '01' and '03' and
strftime('%m', OrderDate) between '01' and '02'
;

 * sqlite:///../data/AdventureWorks.db
Done.


OrderDate,order_revenue,total_revenue
2020-01-01,3455.78,3455.78
2020-01-02,5627.9,9083.68
2020-01-03,11476.51,20560.19
2020-02-01,5913.86,5913.86
2020-02-02,7320.83,13234.689999999999
2020-02-03,9918.75,23153.44
2021-01-01,4961.37,4961.37
2021-01-02,8105.99,13067.36
2021-01-03,5719.0,18786.36
2021-02-01,7599.79,7599.79
